# Adult Income Dataset — Exploratory Data Analysis

This notebook explores the data used to design the preprocessing steps in `src/preprocess.py`.

The goal is not to build a model here. That happens in `src/train.py`. The goal here is to explain why the cleaning steps are needed. This includes checking missing values, feature distributions, class imbalance, and how these affect encoding and evaluation choices.

**Note on the data used here:** download the Adult Income dataset from https://archive.ics.uci.edu/dataset/2/adult and save it as `data/adult.csv` (as described in the README), then update the path in the loader below.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from preprocess import (
    COLUMN_NAMES, NUMERIC_COLS, CATEGORICAL_COLS,
    load_raw_data, clean_data, split_features_target, build_preprocessor,
)

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (8, 4)

## 1. Load the raw data

In [ ]:
df_raw = load_raw_data("../data/adult.csv")
print(f"Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
df_raw.info()

## 2. Missing values

The Adult Income dataset marks missing values with `"?"` rather than a true null, so `df.isna()` won't catch them. They have to be checked for explicitly.

In [ ]:
question_mark_counts = (df_raw == "?").sum()
question_mark_counts = question_mark_counts[question_mark_counts > 0].sort_values(ascending=False)
question_mark_counts

In [ ]:
missing_pct = (question_mark_counts / len(df_raw) * 100).round(2)
missing_pct

**Decision:** Rows with missing values in `workclass`, `occupation`, or `native-country` are dropped instead of filled in. Missing values are rare (under ~5%) and mostly in categorical columns. This is handled in `clean_data()`.

## 3. Duplicate rows

In [ ]:
dupe_count = df_raw.duplicated().sum()
print(f"Exact duplicate rows: {dupe_count} ({dupe_count / len(df_raw):.2%} of the dataset)")

## 4. Target variable: class balance

In [ ]:
target_counts = df_raw["income"].value_counts()
target_pct = df_raw["income"].value_counts(normalize=True) * 100

print(target_counts)
print()
print(target_pct.round(2))

In [ ]:
target_counts.plot(kind="bar", color=["#4C72B0", "#DD8452"])
plt.title("Income class distribution")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Why this matters:** About two-thirds of the data is `<=50K`, so a model that always predicts the majority class would still get high accuracy. Because of this, we also use precision, recall, F1, and ROC-AUC, and choose the best MLflow run based on AUC.

## 5. Numeric feature distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, NUMERIC_COLS):
    ax.hist(df_raw[col], bins=40, color="#4C72B0")
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
df_raw[NUMERIC_COLS].describe()

**Decision:** We apply `StandardScaler` to all numeric columns. Even though `capital-gain` and `capital-loss` are highly skewed, tree-based models (used in most of the tested configurations) are not affected by scaling.

## 6. Categorical feature cardinality

In [ ]:
cardinality = {col: df_raw[col].nunique() for col in CATEGORICAL_COLS}
pd.Series(cardinality).sort_values(ascending=False)

**Decision:** `OneHotEncoder(handle_unknown="ignore")` is used so that unseen categories at inference time do not break the pipeline.

## 7. Relationship between features and the target

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

df_raw.boxplot(column="education-num", by="income", ax=axes[0])
axes[0].set_title("Education-num by income")
axes[0].set_xlabel("")

df_raw.boxplot(column="hours-per-week", by="income", ax=axes[1])
axes[1].set_title("Hours/week by income")
axes[1].set_xlabel("")

df_raw.boxplot(column="age", by="income", ax=axes[2])
axes[2].set_title("Age by income")
axes[2].set_xlabel("")

plt.suptitle("")
plt.tight_layout()
plt.show()

## 8. Running the actual cleaning pipeline

In [ ]:
df_clean = clean_data(df_raw)

print(f"Rows before cleaning: {len(df_raw)}")
print(f"Rows after cleaning:  {len(df_clean)}")
print(f"Rows dropped:         {len(df_raw) - len(df_clean)} "
      f"({(len(df_raw) - len(df_clean)) / len(df_raw):.2%})")

In [ ]:
df_clean["income"].value_counts(normalize=True)

## 9. Fitting the preprocessor

In [ ]:
X, y = split_features_target(df_clean)

preprocessor = build_preprocessor()
preprocessor.fit(X)
X_transformed = preprocessor.transform(X)

print(f"Input shape:  {X.shape}")
print(f"Output shape: {X_transformed.shape}")
print(f"Output dtype: {X_transformed.dtype}")

## Summary of preprocessing decisions

| Decision | Reason |
|---|---|
| Drop rows with `"?"` in workclass/occupation/native-country | Missing values are rare (~5%), so dropping is simpler than imputing |
| Drop duplicate rows | Prevents repeated data from biasing training |
| `StandardScaler` for all numeric columns | Keeps preprocessing simple; tree models are not sensitive to scaling |
| `OneHotEncoder(handle_unknown="ignore")` | Prevents errors from unseen categories at inference |
| `class_weight="balanced"` for Logistic Regression and Random Forest | Handles class imbalance (~65/35 split) |
| Use accuracy, precision, recall, F1, ROC-AUC | Accuracy alone is misleading on imbalanced data |